<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/01_nivelacion_ml/14_clases_desbalanceadas.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Clases desbalanceadas

**Pregunta guía:** ¿Cómo detectar eventos raros sin engañarnos con accuracy?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## El problema

Simulamos un detector donde sólo 2 % son eventos señal. Predecir siempre
“fondo” logra cerca de 98 % de accuracy y utilidad científica nula.
Precision–recall responde preguntas distintas: de las alarmas, ¿cuántas
son reales?, y de las señales, ¿cuántas recuperamos?

El remuestreo debe ocurrir **dentro de cada pliegue de entrenamiento**.
Hacer SMOTE antes de separar replica información hacia validación.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    precision_recall_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
X, y = make_classification(
    n_samples=4_000,
    n_features=15,
    n_informative=7,
    n_redundant=4,
    weights=[0.98, 0.02],
    class_sep=1.0,
    random_state=SEMILLA,
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)
print(pd.Series(y).value_counts(normalize=True).rename("fracción"))


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
estrategias = {
    "dummy": Pipeline([("modelo", DummyClassifier(strategy="most_frequent"))]),
    "pesos": Pipeline(
        [
            ("escala", StandardScaler()),
            ("modelo", LogisticRegression(class_weight="balanced", max_iter=3000)),
        ]
    ),
    "oversampling": Pipeline(
        [
            ("escala", StandardScaler()),
            ("muestreo", RandomOverSampler(random_state=SEMILLA)),
            ("modelo", LogisticRegression(max_iter=3000)),
        ]
    ),
    "SMOTE": Pipeline(
        [
            ("escala", StandardScaler()),
            ("muestreo", SMOTE(random_state=SEMILLA)),
            ("modelo", LogisticRegression(max_iter=3000)),
        ]
    ),
}
filas, ajustados = [], {}
for nombre, modelo in estrategias.items():
    grilla = {"modelo__C": [0.1, 1, 10]} if nombre != "dummy" else {}
    búsqueda = GridSearchCV(
        modelo, grilla, scoring="average_precision", cv=cv, n_jobs=-1
    ).fit(X_dev, y_dev)
    ajustados[nombre] = búsqueda.best_estimator_
    prob = búsqueda.predict_proba(X_test)[:, 1]
    pred = búsqueda.predict(X_test)
    filas.append(
        {
            "estrategia": nombre,
            "AP_test": average_precision_score(y_test, prob),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        }
    )
display(pd.DataFrame(filas).set_index("estrategia"))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for nombre, modelo in ajustados.items():
    prob = modelo.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ax.plot(recall, precision, label=nombre)
ax.axhline(y_test.mean(), color="k", linestyle="--", label="prevalencia")
ax.set(xlabel="recall", ylabel="precision", title="Curvas precision–recall")
ax.legend()
plt.show()

mejor = ajustados["pesos"]
prob = mejor.predict_proba(X_test)[:, 1]
umbrales = np.linspace(0.05, 0.95, 19)
tabla = []
for u in umbrales:
    pred = (prob >= u).astype(int)
    tabla.append({"umbral": u, "alarmas": pred.sum(), "recall": ((pred == 1) & (y_test == 1)).sum() / (y_test == 1).sum()})
display(pd.DataFrame(tabla))


El umbral se escoge en validación según el costo científico de falsos
positivos y falsos negativos; la tabla de test es sólo demostrativa.

**Ejercicios:** imponga recall mínimo 0.90 usando predicciones fuera de
pliegue; compare calibración; explique cuándo SMOTE sería físicamente
absurdo porque interpola entre eventos que no admiten estados intermedios.
